# Pandas Time Series: Vehicle Sales

Work with the FRED monthly Light Weight Vehicle Sales series (`LTOTALNSA`), measured in thousands of vehicles and not seasonally adjusted. Run `python download_data.py` once if the local CSV is missing.

## 1. Load and inspect dates
Parse the observation date and use it as a sorted `DatetimeIndex`. The input series is monthly.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from time_series import (
    add_change_measures,
    load_sales,
    missing_months,
    make_frequency_examples,
    resample_sales,
    rolling_mean,
    save_analysis,
    seasonal_summary,
)

sales = load_sales()
rolling_window = 12
print(type(sales.index).__name__, sales.index.min(), sales.index.max())
sales.head()

## 2. Date ranges and frequencies
`D` creates daily dates and `W` creates weekly dates. Month-end frequency is `ME` in current Pandas; the previous `M` alias is deprecated in Pandas 2.2 and later.

In [ ]:
examples = make_frequency_examples('2024-01-01', '2024-03-31')
for label, dates in examples.items():
    print(f'{label}: {len(dates)} dates; first={dates[0]}, last={dates[-1]}')

## 3. Resample to monthly and quarterly totals
The source observations are already monthly sales in thousands of units. Month-end resampling preserves each month's value, while quarterly resampling sums the three monthly observations. `min_count=1` keeps a completely missing period as missing instead of reporting a misleading zero.

In [ ]:
monthly, quarterly = resample_sales(sales)
display(monthly.head())
display(quarterly.head())

## 4. Shift, difference, and percentage change
`shift(1)` and `diff()` compare each observation with the prior month. A 12-period shift compares it with the same month one year earlier. The seasonal table summarizes observed values by calendar month; the gap report avoids silently filling missing months.

In [ ]:
changes = add_change_measures(sales)
changes.head(15)

seasonal = seasonal_summary(sales)
display(seasonal)

gaps = missing_months(sales)
print(f"Missing calendar months: {len(gaps)}")
display(gaps.head())

## 5. Rolling mean and visualization
The rolling window is configurable. It is trailing and only produces a value after a complete window of observations.

In [ ]:
sales_rolling_mean = rolling_mean(sales, window=rolling_window)
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(sales.index, sales, label='Monthly vehicle sales', color='#2f6fed', linewidth=1.4)
ax.plot(sales_rolling_mean.index, sales_rolling_mean, label=f'{rolling_window}-month rolling mean', color='#ef7c32', linewidth=2.2)
ax.set_title('U.S. Light Weight Vehicle Sales')
ax.set_xlabel('Date')
ax.set_ylabel('Thousands of vehicles (not seasonally adjusted)')
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

## 6. Save reusable outputs
The command-line script saves the same monthly, quarterly, and change tables along with the plot.

In [ ]:
save_analysis(sales, rolling_window=rolling_window)
print('Saved monthly/quarterly totals, changes, seasonal and gap summaries, and charts under outputs/.')